# PS3 Career Readiness: exploration
Run from repo root. Reproduces the corruption report, cleaning, and the no-signal audit.

In [ ]:
import sys; sys.path.insert(0, '..')
import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt
from src.preprocess import clean, CATEGORICAL_VOCAB, TARGET
raw = pd.read_csv('../data/raw/ps3.csv', dtype=str, keep_default_na=False)
raw.head()

## 1. What corruption looks like

In [ ]:
for c in ['Age','Education_Level','Annual_Salary_USD', TARGET]:
    print(c, raw[c].value_counts().head(12).to_dict(), '
')

## 2. Clean and report

In [ ]:
df, report = clean(raw)
print(len(df), 'rows kept')
report

In [ ]:
sns.heatmap(report.drop(index='ID')/len(raw)*100, annot=True, fmt='.0f', cmap='rocket_r'); plt.title('% rows affected');

## 3. Is there any signal?

In [ ]:
num = df.select_dtypes('number').drop(columns='ID')
sns.heatmap(num.corr(), annot=True, fmt='.2f', cmap='vlag', center=0, vmin=-.2, vmax=.2);

In [ ]:
df.groupby(TARGET)[num.columns].mean().round(2)

In [ ]:
from scipy.stats import chi2_contingency
for c in [k for k in CATEGORICAL_VOCAB if k != TARGET]:
    print(c, 'p =', round(chi2_contingency(pd.crosstab(df[c], df[TARGET]))[1], 3))

## 4. Model vs chance
See `python -m src.train` for the full cross-validated audit and permutation test; `reports/metrics.json` has the numbers.

In [ ]:
import json; pd.DataFrame(json.load(open('../reports/metrics.json'))['models']).T